In [1]:
import sys
import glob
import yaml
import pickle
import os
import awkward as ak
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
plt.style.use("~/evanstyle.mplstyle")
sys.path.append("../")
import Utilities as Util

import DataReduction 
import CryoAsicAnalysis


In [16]:
#change path!
#input_path = "/p/lustre1/nexouser/data/StanfordData/ChargeModule/LXe_Run1/Gamma_Data_Post_Surgery_7_15_24/"
input_path = "/Volumes/CODEDRIVE/Gamma_Data_7_16_24/prereduced/6g24pt_sig/"
#change path!
#output_path = "/p/lustre2/nexouser/data/StanfordData/angelico/LXe_Run1_Processed_Data/Gamma_Data_Post_Surgery_7_15_24/Processed_Data/"
#output_path = "../../../data/MockTileRun1/Gamma_Data_Post_Surgery_7_15_24/reduced/"
output_path = "/Volumes/CODEDRIVE/Gamma_Data_7_16_24/reduced/temp/"

input_files = glob.glob(input_path+"*.p")
try:
	print(input_files[0])
except:
	print("No files found in input directory")
	print("Does directory exist?: ", os.path.isdir(input_path))

/Volumes/CODEDRIVE/Gamma_Data_7_16_24/prereduced/6g24pt_sig/Gamma_Data_5kV_7_16_24_initcryo_4_6g_24pt_1000us_file200.p


In [50]:
config_path = "../config/gamma-post-surg-24.yml"
#either pass this directly to the class, or load it and modify with gain/pt settings corrected. 

#with open(config_path, 'r') as f:
#    config = yaml.safe_load(f)

#config["gain"] = 6 


In [52]:
#initialize the DataReduction class
dr = DataReduction.DataReduction(config_path)
#load input data of many files, which combines the dataframes into one
for i, infile in enumerate(input_files):
	print("Reducing file {}".format(infile))
	dr.load_prereduced_data(infile)
	dr.reduce_to_pulses() #does basic initial waveform processing and creates Pulse objects
	dr.process_pulses() #analyzes the pulses in detail to populate pulse reduced quantities
	dr.process_clusters() #clusters pulses into events and measures properties
	dr.process_globals() #measures global properties of the event from the clusters
	dr.dictify_objects() #deletes Pulse and Cluster objects, turning them into dictionaries in the reduced df
	dr.save_reduced_df(output_path, infile.split("/")[-1]) #saves the reduced df to a pickle file


Reducing file /Volumes/CODEDRIVE/Gamma_Data_7_16_24/prereduced/6g24pt_sig/Gamma_Data_5kV_7_16_24_initcryo_4_6g_24pt_1000us_file200.p
loading file /Volumes/CODEDRIVE/Gamma_Data_7_16_24/prereduced/6g24pt_sig/Gamma_Data_5kV_7_16_24_initcryo_4_6g_24pt_1000us_file200.p
Done
Subtracting baselines
Getting full STDs
Analyzing minimums and maximums
Initializing pulses for events with any sample above positive threshold of 4.00 sigma
Got 1003 events with pulses
Rejecting single data point glitches and buffering pulses
Calculating reduced quantities for pulses from 526 remaining events
Processing clusters for 526 events which have pulses
Calculating reduced quantities of clusters for 526 events
Processing global quantities for 526 events which have clusters
Dictifying the Pulse and Cluster objects in prep for data transfer
Reducing file /Volumes/CODEDRIVE/Gamma_Data_7_16_24/prereduced/6g24pt_sig/Gamma_Data_5kV_7_16_24_initcryo_4_6g_24pt_1000us_file201.p
loading file /Volumes/CODEDRIVE/Gamma_Data_

KeyboardInterrupt: 

In [53]:

#combine the reduced files
print("Combining reduced files")
combined_df = pd.DataFrame()
for i, infile in enumerate(glob.glob(output_path+"*.p")):
	if("combined" in infile):
		continue
	df = pickle.load(open(infile, "rb"))[0]
	if(i == 0):
		combined_df = df
	else:
		combined_df = pd.concat([combined_df, df], ignore_index=True)

pickle.dump([combined_df], open(output_path+"combined.p", "wb"))

Combining reduced files
